# Phase 3: Batch Validation - Pipeline Preprocessing Cassava

**Objectif:** Valider le pipeline amélioré sur 1,000 images stratifiées

**Attendu:**
- Taux de succès: ≥95%
- Primary pipeline: ~85%
- Fallback: ~10-15%
- Échecs: <5%

**Inputs:**
- `../data/unified_dataset_manifest.csv` (30,513 images)
- Pipeline amélioré du notebook 03

**Outputs:**
- Métriques de validation
- Images problématiques identifiées
- Rapport de performance par classe

## 1. Setup & Imports

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import time
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
import cv2
import xml.etree.ElementTree as ET

# Torch imports
import torch
from torchvision.ops import box_convert
import supervision as sv

# GroundingDINO
from groundingdino.util.inference import load_model, load_image, predict

# SAM
from segment_anything import sam_model_registry, SamPredictor

print("✓ Imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 2. Configuration

In [ ]:
# Paths
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / 'data'
OUTPUT_DIR = BASE_DIR / 'outputs' / 'batch_validation'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Unified manifest
MANIFEST_PATH = DATA_DIR / 'unified_dataset_manifest.csv'

# Model paths
GROUNDING_DINO_CONFIG = BASE_DIR / 'GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py'
GROUNDING_DINO_CHECKPOINT = BASE_DIR / 'weights/groundingdino_swint_ogc.pth'
SAM_CHECKPOINT = BASE_DIR / 'weights/sam_vit_h_4b8939.pth'

# Processing params
TARGET_SIZE = (448, 448)
PADDING_RATIO = 0.1

# Detection params
TEXT_THRESHOLD = 0.25
PROMPTS = ["cassava leaf", "cassava plant leaf", "green leaf"]

# Class-specific adaptive thresholds
CLASS_THRESHOLDS = {
    0: {'box_threshold': 0.30, 'min_area_ratio': 0.025},  # CBB - standard
    1: {'box_threshold': 0.30, 'min_area_ratio': 0.025},  # CBSD - standard
    2: {'box_threshold': 0.25, 'min_area_ratio': 0.015},  # CGM - MORE PERMISSIVE
    3: {'box_threshold': 0.30, 'min_area_ratio': 0.025},  # CMD - standard
    4: {'box_threshold': 0.28, 'min_area_ratio': 0.020},  # Healthy - slightly permissive
}

# Fallback configuration
FALLBACK_ENABLED = True
FALLBACK_METHOD = 'green_semantic'
GREEN_PERCENTILE_THRESHOLD = 60

# Batch validation params
SAMPLES_PER_CLASS = 200  # 200 * 5 = 1,000 total

# Class names
CLASS_NAMES = {
    0: 'Cassava Bacterial Blight (CBB)',
    1: 'Cassava Brown Streak Disease (CBSD)',
    2: 'Cassava Green Mottle (CGM)',
    3: 'Cassava Mosaic Disease (CMD)',
    4: 'Healthy'
}

print("✓ Configuration loaded")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Validation batch size: {SAMPLES_PER_CLASS} per class = {SAMPLES_PER_CLASS * 5} total")

## 3. Load Models

In [ ]:
print("Loading GroundingDINO model...")
grounding_dino_model = load_model(
    str(GROUNDING_DINO_CONFIG),
    str(GROUNDING_DINO_CHECKPOINT),
    device='cuda' if torch.cuda.is_available() else 'cpu'
)
print("✓ GroundingDINO loaded")

print("\nLoading SAM model...")
sam = sam_model_registry["vit_h"](
    checkpoint=str(SAM_CHECKPOINT)
)
sam.to('cuda' if torch.cuda.is_available() else 'cpu')
sam_predictor = SamPredictor(sam)
print("✓ SAM loaded")

print("\n" + "="*70)
print("MODELS READY")
print("="*70)

## 4. Load Unified Dataset Manifest

In [ ]:
# Load unified manifest
print("Loading unified dataset manifest...")
manifest_df = pd.read_csv(MANIFEST_PATH)

print(f"✓ Manifest loaded: {len(manifest_df):,} images")
print(f"\nDistribution by class:")
class_counts = manifest_df['label'].value_counts().sort_index()
for label, count in class_counts.items():
    print(f"  {label} - {CLASS_NAMES[label]:45s}: {count:,} images")

print(f"\nImages with XML annotations: {manifest_df['has_bbox_annotation'].sum():,}")
print(f"\nSources: {manifest_df['source'].nunique()}")
print(manifest_df['source'].value_counts().to_string())

## 5. Import Pipeline Functions

We'll copy the essential functions from notebook 03.

In [ ]:
# Detection function
def detect_leaves_grounding_dino(image_path, prompts, box_threshold, text_threshold):
    """Detect leaves using GroundingDINO."""
    image_source, image_transformed = load_image(str(image_path))
    
    caption = ". ".join(prompts)
    boxes, confidences, labels = predict(
        model=grounding_dino_model,
        image=image_transformed,
        caption=caption,
        box_threshold=box_threshold,
        text_threshold=text_threshold
    )
    
    h, w, _ = image_source.shape
    boxes = boxes * torch.tensor([w, h, w, h])
    xyxy_boxes = box_convert(boxes=boxes, in_fmt='cxcywh', out_fmt='xyxy').cpu().numpy()
    confidences = confidences.cpu().numpy()
    
    return xyxy_boxes, confidences, labels, image_source

# SAM segmentation
def segment_leaf_sam(image_array, box):
    """Segment leaf using SAM."""
    sam_predictor.set_image(image_array)
    masks, scores, _ = sam_predictor.predict(
        box=box,
        multimask_output=True
    )
    best_mask_idx = np.argmax(scores)
    return masks[best_mask_idx], scores[best_mask_idx]

# Validation
def validate_detection(box, mask, image_shape, min_area_ratio):
    """Validate detection quality."""
    h, w = image_shape[:2]
    total_pixels = h * w
    
    box_w = box[2] - box[0]
    box_h = box[3] - box[1]
    box_area = box_w * box_h
    box_area_ratio = box_area / total_pixels
    
    mask_area = np.sum(mask)
    mask_area_ratio = mask_area / total_pixels
    
    is_valid = box_area_ratio >= min_area_ratio
    
    metrics = {
        'box_area_ratio': box_area_ratio,
        'mask_area_ratio': mask_area_ratio,
        'box_width': box_w,
        'box_height': box_h
    }
    
    return is_valid, metrics

# Green dominance check
def check_green_dominance(image_array, mask, threshold=1.2):
    """Check if masked region is predominantly green."""
    if np.sum(mask) == 0:
        return False, 0.0
    
    masked_pixels = image_array[mask]
    R = masked_pixels[:, 0].astype(np.float32)
    G = masked_pixels[:, 1].astype(np.float32)
    B = masked_pixels[:, 2].astype(np.float32)
    
    green_ratio = np.mean(G / ((R + B) / 2 + 1e-6))
    is_green = green_ratio >= threshold
    
    return is_green, float(green_ratio)

# Crop and resize
def crop_and_resize_leaf(image_array, mask, box, target_size=(448, 448), padding_ratio=0.1):
    """Crop leaf region with padding and resize."""
    h, w = image_array.shape[:2]
    x1, y1, x2, y2 = [int(v) for v in box]
    
    # Add padding
    pad_w = int((x2 - x1) * padding_ratio)
    pad_h = int((y2 - y1) * padding_ratio)
    
    x1_pad = max(0, x1 - pad_w)
    y1_pad = max(0, y1 - pad_h)
    x2_pad = min(w, x2 + pad_w)
    y2_pad = min(h, y2 + pad_h)
    
    cropped = image_array[y1_pad:y2_pad, x1_pad:x2_pad]
    cropped_mask = mask[y1_pad:y2_pad, x1_pad:x2_pad]
    
    resized = cv2.resize(cropped, target_size, interpolation=cv2.INTER_LINEAR)
    resized_mask = cv2.resize(cropped_mask.astype(np.uint8), target_size, interpolation=cv2.INTER_NEAREST)
    
    return resized, resized_mask

print("✓ Helper functions defined")